# klygo.archive (context.py): ArchiveFile context & object API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Context.ipynb)


This notebook favors runnable workflows over long reference tables. It uses only generated data inside a resettable temporary directory.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample data

In [1]:
from tempfile import TemporaryDirectory
from zipfile import BadZipFile
from klygo import archive, files

if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-archive-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-archive-hdxsrg66


In [2]:
source = files.mkdir(root / "dataset")
files.save(source / "labels.txt", ["cat", "dog"], verbose=False)
files.save(source / "config.json", {"batch": 16, "classes": 2}, verbose=False)
images = files.mkdir(source / "images")
files.save(images / "cat.txt", "image placeholder", verbose=False)

sample_archive = root / "dataset.zip"
archive.compress(source, sample_archive, verbose=False)
print("Members:", archive.list_files(sample_archive))

Members: ['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


## Jump to an API

[`open`](#open) · [`ArchiveFile`](#archivefile)

<a id="open"></a>

## `archive.open`

Create an ``ArchiveFile`` context-manager wrapper.

**Signature:** `archive.open(archive_path: Union[str, pathlib.Path]) -> klygo.archive.context.ArchiveFile`

Returns the public `ArchiveFile` wrapper and can be used as a context manager.

Each block below is runnable in order.

**Basic use**

In [3]:
with archive.open(sample_archive) as opened:
    print(opened.format)
    print(opened.list_files())

zip
['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


**Practical variation**

In [4]:
with archive.open(sample_archive) as opened:
    print(opened.search("*.json"))
    print(opened.get_info()["file_count"])
    print(opened.test())

['dataset/config.json']
3
True


**Expected error or edge behavior**

In [5]:
try:
    archive.open(root / "unsupported.txt")
except ValueError as error:
    print("ValueError:", error)

ValueError: Unsupported or unrecognized archive format for file 'C:\Users\Mine\AppData\Local\Temp\klygo-archive-hdxsrg66\unsupported.txt'. Supported formats: ['7z', 'gz', 'rar', 'tar', 'tar.bz2', 'tar.gz', 'tar.xz', 'zip']


<a id="archivefile"></a>

## `archive.ArchiveFile`

Object-oriented interface to a single archive.

**Signature:** `archive.ArchiveFile(archive_path: Union[str, pathlib.Path])`

Binds one archive path and backend for repeated operations. The context manager does not hold an archive stream open between calls.

Each block below is runnable in order.

**Basic use**

In [6]:
opened = archive.ArchiveFile(sample_archive)
print(opened.archive_path)
print(opened.format)
print(list(opened.iter_files()))

C:\Users\Mine\AppData\Local\Temp\klygo-archive-hdxsrg66\dataset.zip
zip
['dataset/config.json', 'dataset/images/cat.txt', 'dataset/labels.txt']


**Practical variation**

In [7]:
object_output = root / "object-output"
opened.extract_file(
    next(name for name in opened.list_files() if name.endswith("config.json")),
    object_output,
)
print([files.name(path) for path in files.find(object_output)])

['config.json']


**Expected error or edge behavior**

In [8]:
try:
    archive.ArchiveFile(root / "missing.unknown")
except ValueError as error:
    print("ValueError:", error)

ValueError: Unsupported or unrecognized archive format for file 'C:\Users\Mine\AppData\Local\Temp\klygo-archive-hdxsrg66\missing.unknown'. Supported formats: ['7z', 'gz', 'rar', 'tar', 'tar.bz2', 'tar.gz', 'tar.xz', 'zip']


## Put it together

In [9]:
with archive.open(sample_archive) as handle:
    json_members = handle.search("*.json")
    extracted_cfg = root / "context_extracted"
    handle.extract_file(json_members[0], extracted_cfg, overwrite=True)
    print("Extracted via ArchiveFile:", [files.name(p) for p in files.find(extracted_cfg)])

Extracted via ArchiveFile: ['config.json']


## Remember

- Use `archive.open(path)` or `archive.ArchiveFile(path)` to bind an archive path once when running multiple inspection, search, and extraction calls.
- Next in `klygo.archive`: start over at [Document_Archive_Io.ipynb](Document_Archive_Io.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Io.ipynb)) or return to [Document_Archive_Overview.ipynb](Document_Archive_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/archive/Document_Archive_Overview.ipynb)).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).